# 2. Area weighting and missing coverage

**SYNTHETIC TEACHING DATA.** This exercise examines the assumptions behind totals rather than identifying a best population product. Run all cells; only bundled local files and the Python standard library are required.

Population counts are summed after allocating a partial cell by its spherical area fraction: `cell_count × overlap_area / cell_area`. This assumes uniform population inside each cell. The method does not recover unknown within-cell settlement patterns. Density inputs require a documented area conversion and are not accepted by this count-only exercise.


In [ ]:
from population_tools import load_demo, compare, area
demo = load_demo()
whole = compare(demo['layers'], demo['bounds'])
for result in whole:
    print(result['metadata']['id'], result['population'], result['coverage'], 'SYNTHETIC')
assert [round(r['population']) for r in whole] == [1600, 1600, 1600, 960]


## A partial cell with an independent expected value

Take half of the first uniform cell along longitude. Its spherical area is exactly half of the cell's area, so its count contribution is 50. A north/south cut requires the sine-of-latitude ratio, not a flat latitude ratio.


In [ ]:
uniform = demo['layers'][0]
half = compare([uniform], [120, 14, 120.05, 14.1])[0]
assert abs(half['population'] - 50) < 1e-7
assert abs(half['coverage'] - 1) < 1e-10
print(half)


## Zero is not NoData

The synthetic daytime scenario has a deliberately empty southern row with valid zero counts. The incomplete-coverage scenario has no northeast cells at all. Missing coverage must not be silently added to the zero-population category. Also inspect how an area extending beyond all cells reduces coverage.


In [ ]:
zero = compare([demo['layers'][2]], [120, 14, 120.1, 14.1])[0]
missing = compare([demo['layers'][3]], [120.2, 14.2, 120.4, 14.4])[0]
assert zero['population'] == 0 and zero['coverage'] > .999999
assert missing['population'] is None and missing['coverage'] == 0
print('Covered zero:', zero['population'], 'coverage:', zero['coverage'])
print('Missing:', missing['population'], 'coverage:', missing['coverage'])
print('Beyond the grid:', compare([uniform], [119.9, 13.9, 120.1, 14.1])[0]['coverage'])


## Data and software credit

JupyterLite-Population-Data-Comparison contributors (2026). *Invented comparison cells, version 1.0*. CC0-1.0: https://creativecommons.org/publicdomain/zero/1.0/. Generated by deterministic arithmetic, with no census, imagery, boundary, or population data inputs. Full per-layer metadata are in `demo.json`; the repository's `data/README.md` records the checksum. `population_tools.py` calculates spherical area-weighted count fractions and preserves source metadata.

Execution: Project Jupyter's JupyterLite (https://jupyterlite.readthedocs.io/) and Pyodide (https://pyodide.org/). Explanations are rendered with Jupyter Book (https://jupyterbook.org/). These dependencies retain their upstream licenses. The standalone viewer uses Leaflet 1.9.4 (BSD-2-Clause).

RustGIS, GeoLibre, JupyterGIS and a FieldWork widget remain future integration work. This exercise exports supported text/GeoJSON artifacts and does not claim native RustGIS-in-Pyodide or GeoTIFF support.
